# YOLOv11 Video-Based Detection on UA-DETRAC (Modular Edition)

Notebook này chạy mô hình YOLO đã train trực tiếp trên các **file video (.mp4)** của UA-DETRAC, trích xuất tọa độ hộp nhận diện của các xe và lưu thành các file `.txt` phục vụ cho việc chạy thuật toán **SORT**.

## 📂 Lợi ích của việc xử lý theo Video:
- **Tiết kiệm dung lượng cực lớn**: Video nén hiệu quả hơn hàng ngàn ảnh JPEGs đơn lẻ, giúp bạn tải lên/tải xuống và giải nén trên Kaggle nhanh gấp hàng chục lần.
- **Tốc độ đọc nhanh hơn**: Đọc frame từ luồng video bằng OpenCV nhanh hơn rất nhiều so với việc đọc hàng ngàn file ảnh riêng lẻ từ ổ cứng.

## ⚡ Quy trình hoạt động:
1. Quét thư mục chứa video để tìm các file `.mp4` tập Test (ví dụ: `MVI_39031.mp4`).
2. Mở từng video, đọc tuần tự các frame ảnh bằng `cv2.VideoCapture`.
3. Chạy model YOLO đã train trên từng frame.
4. Tải file XML tương ứng (ví dụ: `MVI_39031.xml`) để lọc bỏ các nhận diện trùng với **Ignored Regions**.
5. Lưu kết quả thành các file nhãn `.txt` phẳng trong thư mục `./all_predictions` dưới tên: `<video_name>_img<frame_number>.txt` (Sẵn sàng 100% cho SORT).

In [ ]:
# Tự động cài đặt Ultralytics (YOLO) nếu chưa có trong môi trường Kaggle
try:
    import ultralytics

    print("✓ Thư viện Ultralytics đã được cài đặt sẵn!")
except ImportError:
    print("⚠️ Không tìm thấy Ultralytics. Đang tiến hành cài đặt tự động...")
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    print("✓ Cài đặt Ultralytics thành công!")


In [ ]:
import os
import re
import sys
from pathlib import Path
from typing import List
import xml.etree.ElementTree as ET

import cv2
import numpy as np
from tqdm.auto import tqdm
from ultralytics import YOLO

print("✓ Khởi tạo môi trường YOLO thành công!")


## 1. Các hàm trợ giúp và Bộ lọc Ignored Regions

In [ ]:
def find_annotation_xml(sequence_name: str, annotations_folder: str = None) -> str:
    """Tìm file XML ground-truth chứa thông tin Ignored Region."""
    if not sequence_name or not annotations_folder:
        return None
    candidates = [
        os.path.join(annotations_folder, f"{sequence_name}.xml"),
        os.path.join(annotations_folder, sequence_name, f"{sequence_name}.xml"),
    ]
    for path in candidates:
        if os.path.isfile(path):
            return os.path.abspath(path)
    return None


def load_detrac_ignored_regions_xml(xml_path: str) -> List[np.ndarray]:
    """Đọc các hộp Ignored Region từ file XML."""
    ignored_regions = []
    if not xml_path or not os.path.isfile(xml_path):
        return ignored_regions

    root = ET.parse(xml_path).getroot()
    ignored_node = root.find("ignored_region")
    if ignored_node is None:
        return ignored_regions

    for box_node in ignored_node.findall("box"):
        left = float(box_node.attrib["left"])
        top = float(box_node.attrib["top"])
        width = float(box_node.attrib["width"])
        height = float(box_node.attrib["height"])
        ignored_regions.append(
            np.array([left, top, left + width, top + height], dtype=float)
        )

    return ignored_regions


def bbox_area(bbox: np.ndarray) -> float:
    return max(0.0, float(bbox[2] - bbox[0])) * max(0.0, float(bbox[3] - bbox[1]))


def intersection_area(box_a: np.ndarray, box_b: np.ndarray) -> float:
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def ignored_overlap_ratio(bbox: np.ndarray, ignored_regions: List[np.ndarray]) -> float:
    area = bbox_area(bbox)
    if area <= 0.0 or not ignored_regions:
        return 0.0
    ignored_area = sum(intersection_area(bbox, region) for region in ignored_regions)
    return min(1.0, ignored_area / area)


def filter_detections_by_ignore(
    detections: np.ndarray, ignored_regions: List[np.ndarray], overlap_threshold: float
) -> np.ndarray:
    if detections.size == 0 or not ignored_regions:
        return detections
    keep = [
        ignored_overlap_ratio(det[:4], ignored_regions) < overlap_threshold
        for det in detections
    ]
    return detections[np.array(keep, dtype=bool)] if keep else detections


def save_yolo_txt(txt_path: Path, detections: np.ndarray, img_w: int, img_h: int):
    """Lưu kết quả phát hiện ra file text chuẩn định dạng YOLO."""
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for det in detections:
        x1, y1, x2, y2, conf, cls_id = det.tolist()
        xc = ((x1 + x2) / 2.0) / img_w
        yc = ((y1 + y2) / 2.0) / img_h
        w = (x2 - x1) / img_w
        h = (y2 - y1) / img_h
        lines.append(f"{int(cls_id)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f} {conf:.6f}")
    txt_path.write_text("\n".join(lines), encoding="utf-8")


## 2. Cấu hình Đường dẫn trên Kaggle

In [ ]:
# === CẤU HÌNH ĐƯỜNG DẪN KAGGLE ===

# 1. Thư mục chứa các file video .mp4 của UA-DETRAC
VIDEOS_FOLDER = "/kaggle/input/datasets/bratjay/ua-detrac-orig/ua_detrac_test_videos"
if not os.path.exists(VIDEOS_FOLDER):
    VIDEOS_FOLDER = "./ua_detrac_test_videos"  # Fallback local

# 2. Thư mục chứa XML Annotations Test (Dùng để lọc Ignored Regions)
ANNOTATIONS_FOLDER = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML"

# 3. Thư mục lưu các file detections đầu ra dạng .txt (cho SORT đọc)
OUTPUT_TXT_DIR = "/kaggle/working/all_predictions"

# 4. File weight mô hình YOLOv11 đã train của bạn
WEIGHTS_PATH = "/kaggle/input/weights-yolo/yolov11n_last.pt"
if not os.path.exists(WEIGHTS_PATH):
    WEIGHTS_PATH = "./yolov11n_last.pt"  # Fallback weight local nếu có

# --- THAM SỐ PHÁT HIỆN ---
IMG_SIZE = 640
CONF_THRESHOLD = 0.30
IOU_THRESHOLD = 0.50
FRAME_STRIDE = 4  # Nhảy cóc mỗi bao nhiêu frame (Nếu muốn chạy tất cả chỉnh thành 1)

# --- THAM SỐ IGNORE ---
FILTER_IGNORED_REGIONS = True
IGNORE_OVERLAP_THRESHOLD = 0.50

# --- TỰ ĐỘNG PHÁT HIỆN PATH TRÊN KAGGLE (ĐỀ PHÒNG SAI KHÁC) ---
kaggle_input = Path("/kaggle/input")
if kaggle_input.is_dir():
    # Tìm kiếm tự động file weights.pt
    weights_pt = list(kaggle_input.rglob("yolov11n_last.pt"))
    if weights_pt:
        WEIGHTS_PATH = str(weights_pt[-1])

    # Tìm kiếm tự động thư mục videos
    videos_dir = list(kaggle_input.rglob("ua_detrac_test_videos"))
    if videos_dir:
        VIDEOS_FOLDER = str(videos_dir[-1])

    # Tìm kiếm tự động thư mục XML Annotations
    ann_dir = list(kaggle_input.rglob("DETRAC-Test-Annotations-XML"))
    if ann_dir:
        ANNOTATIONS_FOLDER = str(ann_dir[-1])

print("✓ Thiết lập cấu hình thành công!")
print(f"Weights Path: {WEIGHTS_PATH}")
print(f"Videos Folder: {VIDEOS_FOLDER}")
print(f"XML Annotations: {ANNOTATIONS_FOLDER}")
print(f"Output Flat TXT: {OUTPUT_TXT_DIR}")


## 3. Thực thi phát hiện và Lưu Nhãn Detections

In [ ]:
from pathlib import Path

out_path_dir = Path(OUTPUT_TXT_DIR)
out_path_dir.mkdir(parents=True, exist_ok=True)

if not os.path.exists(WEIGHTS_PATH):
    raise FileNotFoundError(f"❌ Không tìm thấy file weights tại: {WEIGHTS_PATH}")
if not os.path.isdir(VIDEOS_FOLDER):
    raise FileNotFoundError(f"❌ Không tìm thấy thư mục video tại: {VIDEOS_FOLDER}")

# Khởi tạo mô hình YOLO
model = YOLO(WEIGHTS_PATH)
print(f"✓ Đã tải mô hình YOLO từ: {WEIGHTS_PATH}")

# Quét các file video .mp4 đầu vào
video_paths = sorted(list(Path(VIDEOS_FOLDER).glob("*.mp4")))
print(f"[*] Tìm thấy {len(video_paths)} videos để chạy nhận diện.")

total_frames_processed = 0
total_detections_saved = 0

for i, v_path in enumerate(video_paths):
    seq_name = v_path.stem  # Tên sequence (ví dụ: MVI_39031)
    print(f"\n[{i+1}/{len(video_paths)}] Đang xử lý video: {v_path.name}")

    # Tải Ignored Regions của sequence này
    xml_path = find_annotation_xml(seq_name, ANNOTATIONS_FOLDER)
    ignored_regions = load_detrac_ignored_regions_xml(xml_path) if xml_path else []
    if ignored_regions:
        print(f"   [i] Lọc {len(ignored_regions)} ignored regions dựa theo file XML.")

    # Mở video
    cap = cv2.VideoCapture(str(v_path))
    if not cap.isOpened():
        print(f"   ❌ Lỗi: Không thể mở video {v_path.name}")
        continue

    # Lấy thông tin video
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"   [i] FPS: {fps:.2f} | Tổng số frame: {total_frames}")

    frame_idx = 0

    # Vòng lặp đọc từng frame của video
    with tqdm(total=total_frames, desc=f"      {seq_name}", leave=False) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_idx += 1
            pbar.update(1)

            # Chỉ xử lý các frame chia hết cho FRAME_STRIDE
            if (
                frame_idx % FRAME_STRIDE != 0
                and FRAME_STRIDE > 1
                and frame_idx != 1
                and frame_idx != total_frames
            ):
                continue

            img_h, img_w = frame.shape[:2]

            # Dự đoán hộp nhận diện bằng YOLO
            result = model.predict(
                frame,
                imgsz=IMG_SIZE,
                conf=CONF_THRESHOLD,
                iou=IOU_THRESHOLD,
                verbose=False,
            )[0]

            if result.boxes is None or len(result.boxes) == 0:
                detections = np.zeros((0, 6), dtype=float)
            else:
                xyxy = result.boxes.xyxy.cpu().numpy()
                confs = result.boxes.conf.cpu().numpy().reshape(-1, 1)
                cls_ids = result.boxes.cls.cpu().numpy().reshape(-1, 1)
                detections = np.concatenate([xyxy, confs, cls_ids], axis=1).astype(
                    float
                )

            # Lọc bỏ các xe nằm đè lên Ignored Region
            if FILTER_IGNORED_REGIONS and ignored_regions:
                detections = filter_detections_by_ignore(
                    detections, ignored_regions, IGNORE_OVERLAP_THRESHOLD
                )

            # Lưu file label .txt phẳng theo chuẩn định dạng đặt tên
            txt_filename = f"{seq_name}_img{frame_idx:05d}.txt"
            txt_file_path = out_path_dir / txt_filename

            save_yolo_txt(txt_file_path, detections, img_w, img_h)

            total_frames_processed += 1
            total_detections_saved += len(detections)

    cap.release()
    print(f"   ✓ Đã xử lý xong: Tạo ra các file detections trong {out_path_dir.name}")

print("\n" + "=" * 80)
print(f"🎉 Hoàn thành! Đã chạy xong YOLO nhận diện cho tất cả video.")
print(
    f"📊 Thống kê: Xử lý {total_frames_processed} frames, lưu {total_detections_saved} detections."
)
print(f"📂 Đầu ra lưu tại thư mục: {out_path_dir.resolve()}")
print("=" * 80)


## 4. Nén kết quả nhãn để tải về (Tùy chọn)

In [ ]:
import shutil

zip_name = "/kaggle/working/all_predictions_archive"
if os.path.exists(OUTPUT_TXT_DIR) and len(os.listdir(OUTPUT_TXT_DIR)) > 0:
    print(f"📦 Đang nén thư mục nhãn detections {OUTPUT_TXT_DIR}...")
    zip_path = shutil.make_archive(zip_name, "zip", OUTPUT_TXT_DIR)
    print(f"🎉 Đã nén thành công! File lưu tại: {zip_path}")
else:
    print("⚠️ Không có file nhãn nào để nén.")
